# Model Optimization Notebook
This notebook demonstrates post-training optimization steps for a fine-tuned small language model, including quantization, ONNX export, artifact benchmarking, and metadata persistence.

In [ ]:
# 1. Import Libraries
import json
import time
from pathlib import Path
from dataclasses import dataclass
import logging
from typing import List, Dict, Any

from src.training.model_optimizer import (
    quantize_model,
    export_to_onnx,
    summarize_artifact,
    benchmark_inference,
)


In [ ]:
# 2. Load Optimization Config
config_path = Path("configs/optimization_config.yaml")
if not config_path.exists():
    raise FileNotFoundError(f"Missing config: {config_path}")

import yaml
with open(config_path) as f:
    raw_cfg = yaml.safe_load(f)
raw_cfg

In [ ]:
# 3. Quantize Model
base_model_id = raw_cfg["base_model_id"]
quant_cfg = raw_cfg["quantization"]
quant_dir = Path(raw_cfg["output_root"]) / f"{base_model_id.replace('/', '_')}-quant"

if quant_cfg.get("enabled", True):
    quant_result = quantize_model(
        base_model_id,
        quant_dir,
        method=quant_cfg.get("method", "int8"),
        dtype=quant_cfg.get("dtype"),
    )
    quant_meta = summarize_artifact(quant_result)
else:
    quant_result = None
    quant_meta = {"enabled": False}
quant_meta

In [ ]:
# 4. Export ONNX
onnx_cfg = raw_cfg["onnx"]
onnx_path = Path(raw_cfg["output_root"]) / f"{base_model_id.replace('/', '_')}.onnx"
if onnx_cfg.get("enabled", True):
    onnx_result = export_to_onnx(
        base_model_id,
        onnx_path,
        opset=onnx_cfg.get("opset", 17),
        sequence_length=onnx_cfg.get("sequence_length", 128),
    )
    onnx_meta = summarize_artifact(onnx_result)
else:
    onnx_meta = {"enabled": False}
onnx_meta

In [ ]:
# 5. Benchmark Quantized Artifact (if present)
if quant_result:
    bench_stats = benchmark_inference(quant_result.artifact_path, repetitions=raw_cfg["benchmark"]["repetitions"], max_new_tokens=raw_cfg["benchmark"]["max_new_tokens"], prompt=raw_cfg["benchmark"]["prompt"])
else:
    bench_stats = {"skipped": True}
bench_stats

In [ ]:
# 6. Compare Sizes
sizes = {}
if quant_result:
    sizes["quantized_bytes"] = quant_result.size_bytes
if onnx_path.exists():
    sizes["onnx_bytes"] = onnx_path.stat().st_size
sizes

In [ ]:
# 7. Persist Summary
summary = {
    "quantization": quant_meta,
    "onnx": onnx_meta,
    "benchmark": bench_stats,
    "sizes": sizes,
}
summary_path = Path(raw_cfg["output_root"]) / "summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2))
summary_path, summary